In [ ]:
# tools we use in this game
import base64        # to change the Lyria audio data back to mp3
import io            # to work with image data in memory
import os            # to check if a file exists
import random        # to pick random movies and options
import time          # to wait between Lyria requests

import ipywidgets as widgets       # buttons, text box, progress bar
import pandas as pd                # to read the csv files
import requests                    # to download images and call Lyria
from google.auth.transport.requests import Request    # to get the token
from google.oauth2 import service_account              # to read the key file
from IPython.display import display     # to show the widgets
from PIL import Image                   # to open and resize images

In [ ]:
# read the two csv files
movie_df = pd.read_csv("data/movie_data.csv")
images_df = pd.read_csv("data/movie_images.csv", dtype=str)   # dtype=str: read every cell as text

# both files have a "title" column, so we can merge them into one table
# how="left": keep all the movies from movie_df
merged_df = pd.merge(movie_df, images_df, on="title", how="left")

# change the empty cells (NaN) to empty text ""
merged_df = merged_df.fillna("")

# a list of all the movie titles, we use it to make the wrong options
all_titles = list(merged_df["title"])

merged_df

In [ ]:
# every screenshot will be 640 x 360, so the buttons always stay in the same place
FRAME_W = 640
FRAME_H = 360


def fit_to_frame(img):
    # make the image smaller so it can fit in 640 x 360 (the shape does not change)
    img = img.convert("RGB")
    img.thumbnail((FRAME_W, FRAME_H))

    # make a black background, then put the image in the center
    frame = Image.new("RGB", (FRAME_W, FRAME_H), "black")
    x = (FRAME_W - img.width) // 2
    y = (FRAME_H - img.height) // 2
    frame.paste(img, (x, y))

    # save the image as jpeg data (in memory, not as a file)
    image_bytes = io.BytesIO()
    frame.save(image_bytes, format="JPEG")
    return image_bytes.getvalue()


# download all the screenshots one time, before the game starts
# image_cache is a dictionary: movie title -> image data
image_cache = {}
broken = []

for _, movie in merged_df.iterrows():
    try:
        # pretend to be a browser, some websites block python
        response = requests.get(movie["image_url"], headers={"User-Agent": "Mozilla/5.0"}, timeout=10)
        img = Image.open(io.BytesIO(response.content))
        image_cache[movie["title"]] = fit_to_frame(img)
    except Exception:
        # if the link does not work, the game does not crash
        # we just remember the movie, and this movie will only get clue questions
        broken.append(movie["title"])

print("Screenshots that work:", len(image_cache))
print("Broken links:", broken)

In [ ]:
# one Lyria prompt for each sound
SOUND_PROMPTS = {
    "start": "Short upbeat cinematic game show intro, orchestral brass fanfare with drums, exciting and bright, instrumental",
    # 3 ending songs, the game picks one by the score
    "end_low": "Comedic mocking game show fail song, sad trombone wah-wah, bouncy honky-tonk piano, kazoo and slide whistle, teasing and silly, a goofy cartoonish male voice laughs and sings 'You need to watch more movies!', playful roast, funny and light",
    "end_mid": "Bright celebratory congratulations song, upbeat pop orchestral fanfare with brass, handclaps and a cheering crowd, a joyful group of singers sings 'Great job! You know so much about movies!', feel-good award ceremony energy, confetti moment",
    "end_high": "Epic heavenly hymn, majestic cathedral pipe organ, huge angelic choir and soaring orchestra with timpani and bells, glorious, holy and grand, the choir sings 'You are legendary! Hallelujah!' in praise, golden light, triumphant divine finale",
}


def get_audio(prompt, token):
    # this is the same request as the class example
    url = "https://aiplatform.googleapis.com/v1beta1/projects/wpp-cto-smartprod-sandbox/locations/global/interactions"
    payload = {
        "model": "lyria-3-clip-preview",
        "input": [
            {
                "type": "text",
                "text": prompt
            }
        ]
    }
    headers = {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json",
    }
    response = requests.post(url, headers=headers, json=payload)
    data = response.json()

    # if there is no "outputs", something is wrong (for example the key), so show the error
    if "outputs" not in data:
        print("Error:", data)
        return None

    # look for the item that has the audio data, and change it back to mp3
    for item in data["outputs"]:
        if "data" in item:
            return base64.b64decode(item["data"])
    return None

In [ ]:
# load the key and get a token (same as the class example)
creds = service_account.Credentials.from_service_account_file(
    "keys/service.json",
    scopes=["https://www.googleapis.com/auth/cloud-platform"],
)
creds.refresh(Request())
token = creds.token

# make the folder for the sounds (no error if it is already there)
os.makedirs("data/sounds", exist_ok=True)

for name, prompt in SOUND_PROMPTS.items():
    path = f"data/sounds/{name}.mp3"

    # if we already have this sound, skip it (no need to pay again)
    if os.path.exists(path):
        print(name, "already exists")
        continue

    mp3_bytes = get_audio(prompt, token)
    if mp3_bytes:
        # "wb" = write binary, because mp3 is not a text file
        with open(path, "wb") as f:
            f.write(mp3_bytes)
        print(name, "saved")

    # Lyria only allows 10 requests in 1 minute, so wait 7 seconds
    time.sleep(7)

In [ ]:
TOTAL_QUESTIONS = 30    # one game has 30 questions


def make_options(movie):
    # pick 3 wrong titles (not the right answer)
    wrong = random.sample([t for t in all_titles if t != movie["title"]], 3)
    # add the right answer, then mix the order
    options = wrong + [movie["title"]]
    random.shuffle(options)
    return options


def movie_quiz():

    # ---------- make the widgets ----------
    name_input = widgets.Text(placeholder="Type your name", description="Name:")
    start_button = widgets.Button(description="Start", button_style="success")
    info = widgets.HTML("<b style='font-size:18px'>Type your name, then press Start!</b>")
    question_label = widgets.HTML("")

    # the screenshot and the clue card are the same size (640 x 360)
    question_image = widgets.Image(format="jpeg", width=FRAME_W, height=FRAME_H)
    clue_card = widgets.HTML()
    frame = widgets.Box(layout=widgets.Layout(width=f"{FRAME_W}px", height=f"{FRAME_H}px"))

    # 4 option buttons, they are locked until the game starts
    option_buttons = []
    for i in range(4):
        button = widgets.Button(description="", disabled=True,
                                layout=widgets.Layout(width="316px", height="44px"))
        option_buttons.append(button)
    sound_box = widgets.Box()   # the audio player goes in here (you can not see it)

    # all the game data is saved in this dictionary
    game = {"running": False}

    # ---------- small helper functions ----------
    def play_sound(name):
        path = f"data/sounds/{name}.mp3"
        if not os.path.exists(path):
            return
        with open(path, "rb") as f:
            # autoplay=True: start playing right away
            player = widgets.Audio(value=f.read(), format="mp3", autoplay=True, controls=False)
        # put the new player in the box (the old sound stops)
        sound_box.children = [player]

    def stop_sound():
        # take the player out of the box, then the sound stops
        sound_box.children = []

    def show_info(message="", color="black"):
        # the line at the top: "Score: 3 | Correct! ..."
        text = f"Score: {game['score']}"
        if message != "":
            text += f" | <span style='color:{color}'>{message}</span>"
        info.value = f"<b style='font-size:18px'>{text}</b>"

    def show_card(text):
        # a dark card with white text in the middle, same size as a screenshot
        style = (f"width:{FRAME_W}px; height:{FRAME_H}px; background:#222; color:white; "
                 "display:flex; flex-direction:column; justify-content:center; align-items:center; "
                 "text-align:center; padding:30px; box-sizing:border-box; font-size:22px;")
        clue_card.value = f"<div style='{style}'>{text}</div>"
        frame.children = [clue_card]

    def lock_buttons(locked):
        for button in option_buttons:
            button.disabled = locked

    # ---------- the game ----------
    def new_question():
        # after 30 questions, the game is over
        if game["number"] == TOTAL_QUESTIONS:
            end_game()
            return

        # take one movie from the list (the list is already in random order)
        movie = game["movies"].pop()
        game["movie"] = movie
        game["options"] = make_options(movie)
        game["number"] += 1

        # write the 4 titles on the buttons
        for i in range(4):
            title = game["options"][i]
            option_buttons[i].description = title
            # long titles get smaller text, so they can fit in the button
            if len(title) > 40:
                option_buttons[i].style.font_size = "11px"
            elif len(title) > 30:
                option_buttons[i].style.font_size = "12px"
            else:
                option_buttons[i].style.font_size = "14px"

        # 80% screenshot question, 20% clue question (4:1)
        # if the screenshot is broken, it is always a clue question
        number = f"Question {game['number']} / {TOTAL_QUESTIONS}"
        if movie["title"] in image_cache and random.random() < 0.8:
            question_label.value = f"<b>{number}: Which movie is this?</b>"
            question_image.value = image_cache[movie["title"]]
            frame.children = [question_image]
        else:
            question_label.value = f"<b>{number}: Clue</b>"
            show_card(movie["clue"])

    def answer(choice):
        # choice is the number of the button: 0, 1, 2 or 3
        if not game["running"]:
            return
        stop_sound()   # the start music stops when the player answers

        movie = game["movie"]
        if game["options"][choice] == movie["title"]:
            game["score"] += 1
            show_info(f"Correct! {movie['title']} ({movie['year']})", color="green")
        else:
            show_info(f"Wrong! The answer was {movie['title']} ({movie['year']})", color="red")

        new_question()

    def end_game():
        game["running"] = False
        lock_buttons(True)
        for button in option_buttons:
            button.description = " "
        question_label.value = ""

        # pick the ending by the score
        name = game["name"]
        score = game["score"]
        if score <= 10:
            ending = "end_low"
            message = "You need to watch more movies!"
        elif score <= 20:
            ending = "end_mid"
            message = "Great job, you know a lot about movies!"
        else:
            ending = "end_high"
            message = "You are LEGENDARY!"

        show_card(f"<div style='font-size:40px; font-weight:bold'>GAME OVER</div>"
                  f"<div style='font-size:24px; margin-top:10px; color:#ffd54f'>"
                  f"{name}, you got {score} / {TOTAL_QUESTIONS} right!<br>{message}</div>")
        info.value = f"<b style='font-size:18px'>Final score: {score} / {TOTAL_QUESTIONS}</b>"
        play_sound(ending)

        # the next player can play
        start_button.description = "Play again"
        start_button.disabled = False
        name_input.disabled = False

    # ---------- what the buttons do ----------
    def on_start_click(b):
        name = name_input.value.strip()
        if name == "":
            info.value = "<b style='font-size:18px; color:red'>Please type your name first!</b>"
            return

        # all the movies, in random order
        movies = merged_df.to_dict("records")
        random.shuffle(movies)

        # reset all the game data
        game["running"] = True
        game["name"] = name
        game["score"] = 0
        game["number"] = 0
        game["movies"] = movies

        start_button.disabled = True
        name_input.disabled = True
        lock_buttons(False)
        show_info(f"Good luck, {name}!")
        play_sound("start")
        new_question()

    def on_option_click(b):
        # b is the button that was clicked, find its number (0 to 3)
        choice = option_buttons.index(b)
        answer(choice)

    # connect the functions to the buttons
    start_button.on_click(on_start_click)
    for button in option_buttons:
        button.on_click(on_option_click)

    # ---------- show everything ----------
    show_card("Type your name, then press Start")
    display(widgets.VBox([
        widgets.HBox([name_input, start_button]),
        info,
        question_label,
        frame,
        widgets.HBox(option_buttons[:2]),   # button 1 and 2
        widgets.HBox(option_buttons[2:]),   # button 3 and 4
        sound_box,
    ]))

In [ ]:
movie_quiz()